In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
# Data and features
train=pd.read_csv("../data/case1/train1.csv")
test=pd.read_csv("../data/case1/test1.csv")
for df in (train, test):
    df["n"] = df["speed_rpm"] / 1000
    df["Q"] = df["flow_m3_h"]
    df["P"] = df["power_w"]
    df["deltaP"]=df["outlet_pressure_bar"]-df["inlet_pressure_bar"]

X_train, y_train= train[["n","P","deltaP"]], train[["Q"]]
X_test, y_test= test[["n","P","deltaP"]], test[["Q"]]

# Scalling works best for NN
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train) 
X_test_s=scaler.transform(X_test) 
y_scaler = StandardScaler().fit(y_train)




In [3]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_scaler.transform(y_train), dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
xb, yb = next(iter(train_loader))



In [4]:
class Net(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.hidden = nn.Linear(n_features, 16)
        self.out = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden(x))
        return self.out(x)

model = Net(n_features=X_train_t.shape[1])


In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [6]:
n_epochs = 50

for epoch in range(n_epochs):
    total_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()        # 1. reset gradients
        logits = model(xb)           # 2. forward pass: predictions
        loss = loss_fn(logits, yb)   # 3. loss for this batch
        loss.backward()              # 4. backward pass: compute gradients
        optimizer.step()             # 5. update the weights

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: avg loss = {avg_loss:.4f}")

Epoch 1: avg loss = 1.2575
Epoch 2: avg loss = 1.1613
Epoch 3: avg loss = 1.0791
Epoch 4: avg loss = 1.0098
Epoch 5: avg loss = 0.9536
Epoch 6: avg loss = 0.8981
Epoch 7: avg loss = 0.8504
Epoch 8: avg loss = 0.8037
Epoch 9: avg loss = 0.7586
Epoch 10: avg loss = 0.7131
Epoch 11: avg loss = 0.6701
Epoch 12: avg loss = 0.6211
Epoch 13: avg loss = 0.5743
Epoch 14: avg loss = 0.5248
Epoch 15: avg loss = 0.4781
Epoch 16: avg loss = 0.4316
Epoch 17: avg loss = 0.3829
Epoch 18: avg loss = 0.3405
Epoch 19: avg loss = 0.2980
Epoch 20: avg loss = 0.2603
Epoch 21: avg loss = 0.2237
Epoch 22: avg loss = 0.1927
Epoch 23: avg loss = 0.1646
Epoch 24: avg loss = 0.1401
Epoch 25: avg loss = 0.1194
Epoch 26: avg loss = 0.1027
Epoch 27: avg loss = 0.0887
Epoch 28: avg loss = 0.0772
Epoch 29: avg loss = 0.0683
Epoch 30: avg loss = 0.0612
Epoch 31: avg loss = 0.0556
Epoch 32: avg loss = 0.0513
Epoch 33: avg loss = 0.0478
Epoch 34: avg loss = 0.0448
Epoch 35: avg loss = 0.0425
Epoch 36: avg loss = 0.0405
E

In [7]:
# Your solution goes here...
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
model.eval()
with torch.no_grad():
    preds_s = model(X_test_t)

preds = torch.tensor(y_scaler.inverse_transform(preds_s.numpy()), dtype=torch.float32)

rmse = torch.sqrt(torch.mean((preds - y_test_t) ** 2)).item()
mae = torch.mean(torch.abs(preds - y_test_t)).item()
r2 = r2_score(y_test.to_numpy(), preds.numpy())

print(f"Test RMSE: {rmse:.3f} m3/h, MAE: {mae:.3f} m3/h, R2: {r2:.3f}")

Test RMSE: 2.176 m3/h, MAE: 1.620 m3/h, R2: 0.968
